# ATML PA2 - Task 2 (PPO) orchestration on Kaggle (2x T4)

This notebook only **orchestrates** the repository scripts; all experiment code lives in `task2_ppo/`.

**Kaggle settings:** Accelerator = *GPU T4 x2*, Internet = *On*. Use **Save Version -> Save & Run All (Commit)**.
The repository is cloned to `/tmp/repo` so that the notebook output contains **only** `task2_artifacts.zip`
(download it with `kaggle kernels output <user>/<notebook> -p <dest>`).

What runs (see `task2_ppo/run_all.sh`):

| GPU 0 | GPU 1 |
|---|---|
| standard 20-update continuation from the supplied midpoint | cached-batch clipping study (eps = 0.05 / 0.20 / 0.50) |
| held-out eval: SFT reference, midpoint, standard | eps forks 0.05 / 0.20 / 0.50 (8 updates, beta_KL = 0.10) -> eval |
| KL forks beta_KL = 0.0 / 0.20 (8 updates, eps = 0.20) -> eval | |

The eps = 0.20 / beta_KL = 0.10 fork is shared by both studies. Then `task2_ppo.make_report` builds `report/task2/`.

## 1. Get the code

In [ ]:
REPO_URL = "https://github.com/alihasnaink/atml-pa2.git"
BRANCH = "main"
WORKDIR = "/tmp/repo"   # outside /kaggle/working: keeps checkpoints out of the notebook output

import os, subprocess
url = REPO_URL
try:  # optional: private repository
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    url = REPO_URL.replace("https://", f"https://{token}@")
except Exception:
    pass
if not os.path.exists(WORKDIR):
    subprocess.run(["git", "clone", "--branch", BRANCH, url, WORKDIR], check=True)
os.chdir(WORKDIR)
print(subprocess.run(["git", "log", "--oneline", "-3"], capture_output=True, text=True).stdout)

## 2. Install the pinned environment

In [ ]:
!pip install -q -r requirements.txt
!python -m scripts.check_environment
!nvidia-smi --query-gpu=index,name,memory.total --format=csv

## 3. Download and validate the fixed course assets

In [ ]:
!python -m scripts.download_assets
!python -m scripts.validate_assets

## 4. Run all Task 2 experiments
Resumable within a session: re-running this cell skips finished runs/evaluations. Per-GPU logs: `results/task2_ppo/logs/`.

In [ ]:
!bash task2_ppo/run_all.sh

## 5. Quick look at the results

In [ ]:
from IPython.display import Markdown, Image, display
for t in ["heldout_all_conditions", "standard_continuation_compute", "clipping_cached_batch", "cached_batch_reconstruction",
          "clipping_forks", "kl_forks", "paired_vs_midpoint"]:
    p = f"report/task2/tables/{t}.md"
    if os.path.exists(p):
        display(Markdown(open(p).read()))
for f in ["standard_continuation_diagnostics", "clipping_cached_batch", "clipping_forks_trajectories",
          "kl_forks_trajectories", "kl_forks_paired_differences", "reward_vs_kl"]:
    p = f"report/task2/figures/{f}.png"
    if os.path.exists(p):
        display(Image(p))

## 6. Package the artifacts
Unzip `task2_artifacts.zip` at the repository root. It contains `results/task2_ppo/`, `report/task2/` and the LoRA adapters in `outputs/task2_ppo/` (git-ignored; keep `outputs/task2_ppo/standard`, Task 4 uses it).

In [ ]:
!cd /tmp/repo && zip -qr /kaggle/working/task2_artifacts.zip results/task2_ppo report/task2 outputs/task2_ppo
!ls -lh /kaggle/working/